# Spiking neuron

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bernardorivas/hybrid_boxmap/blob/main/notebooks/spiking_neuron.ipynb)

The system is given by

$$100\,\dot v=0.7(v+60)(v+40)-u+70,\qquad \dot u=0.03\,(-2(v+60)-u)$$

on $X=([-80,-40]\times[-300,600])\cup([-40,35]\times[-300,160])$, with guard and reset

$$G=\{(35,u): -300\leq u\leq 160\},\qquad r(35,u)=(-50,u+100).$$

Stored figures are redrawn below. [Recompute from scratch](spiking_neuron_from_scratch.ipynb).


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12 or newer is required; select a current Colab runtime.")

if "google.colab" in sys.modules:
    repo = Path("/content/hybrid_boxmap")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/bernardorivas/hybrid_boxmap.git", str(repo)], check=True)
    # Installation also runs when the checkout already exists.
    if shutil.which("dot") is None:
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "graphviz"], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet", "-e", str(repo),
        "cmgdb==1.3.3+fork.6", "--find-links",
        "https://github.com/bernardorivas/CMGDB/releases/expanded_assets/v1.3.3%2Bfork.6",
    ], check=True)
else:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents, here / "code"]
    repo = next((p for p in candidates if (p / "demo/run_paper_examples.py").is_file()), None)
    if repo is None:
        raise RuntimeError("Open this notebook from the hybrid_boxmap checkout.")

os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
if shutil.which("dot") is None:
    raise RuntimeError("Install Graphviz (the dot executable) before plotting Morse graphs.")
import CMGDB
from hybrid_dynamics.src.carrier_kernel import cmgdb_provenance
if not hasattr(CMGDB, "ComputeCarrierChainMap"):
    raise RuntimeError("Install CMGDB 1.3.3+fork.6 as described in the README.")
print("Repository:", repo)
print("CMGDB:", cmgdb_provenance("native"))


## Phase portrait

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from hybrid_dynamics.examples.paper_examples import paper_problem

problem = paper_problem("spiking-neuron")

fig, ax = plt.subplots(figsize=(5, 4))
for state in [(-75, -250), (-70, 300), (-60, 550), (-45, -200), (0, -280), (20, 100)]:
    trajectory = problem.system.simulate(np.array(state, float), (0.0, 400.0), max_jumps=40, max_step=0.5)
    color = None
    for segment in trajectory.segments:
        line, = ax.plot(*np.asarray(segment.state_values).T, lw=0.8, color=color)
        color = line.get_color()
ax.plot([35, 35], [-300, 160], color="k", lw=2.5)  # guard
ax.set(xlim=(-85, 40), ylim=(-310, 610), xlabel="v", ylabel="u")
plt.show()

## Morse sets

Stored computation: $\tau=5$, level $n=7$, offset $K=3$, $1024^2$ base cells, and 512 phase cells on $R=[-120,200]\times[-400,880]$ (cells contained in $X$).

Base cells per axis are $2^{n+K}$; phase cells are $2^{n+2}$. Gap refinement depth is 14.


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
from IPython.display import Image, display

output_dir = Path("output") / ("notebook-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8])
output_dir.mkdir(parents=True, exist_ok=False)
print("Output:", output_dir.resolve())


In [ ]:
from demo.replot_paper import replot

record = Path("figures/paper/paper-spiking-neuron-tau500-level7-base1024-corners-gap-refined.json")
replot(record, output_dir=output_dir, update_json=False)
stem = output_dir / record.stem
for panel in ['nontrivial-base']:
    path = Path(f"{stem}-{panel}.png")
    if path.exists():
        display(Image(str(path), width=520))


## Conley–Morse graph

Labels give index dimensions in degrees 0, 1, 2. Morse sets with trivial Conley index are gray.

In [ ]:
for panel in ["nontrivial-graph", "graph"]:
    display(Image(f"{stem}-{panel}.png", width=520))
